### Step 1 — Ingestion
Ingestion (Mapping to AWS: S3 -> Bedrock Knowledge Base)

In [14]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("machine-learning-engineer-associate-01.pdf")
docs = loader.load()

### Step 2 — Chunking
Chunking [CERTIFICATION FOCUS: Chunking strategies are heavily tested]

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

### Step 3 — Embeddings & Vector DB
Embeddings & Vector DB (Mapping to AWS: Titan -> OpenSearch)

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_db")
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2075.04it/s]


### Step 4 — LLM Setup
LLM Setup (Mapping to AWS: Bedrock InvokeModel API). Using Groq API with llama-3.1-8b-instant.

In [17]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"), temperature=0.1)

### Step 5 — Orchestration

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know."
    "\n\n{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

rag_chain = (
    {"context": retriever, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)



### Step 6 — Execution

In [20]:
response = rag_chain.invoke("How could I prepare for the certification?")
print(response)


Based on the provided documents, it appears that the AWS Certified Machine Learning Engineer - Associate (MLA-C01) certification is designed for individuals with at least 1 year of experience using Amazon SageMaker and other AWS services for ML engineering. To prepare for the certification, I would recommend the following steps:

1. **Gain relevant experience**: Ensure you have at least 1 year of experience using Amazon SageMaker and other AWS services for ML engineering.
2. **Familiarize yourself with the exam guide**: Study the exam guide provided in the documents, which outlines the topics covered on the exam, including:
	* Choosing deployment infrastructure and endpoints, provisioning compute resources, and configuring auto scaling based on requirements.
	* Setting up continuous integration and continuous delivery (CI/CD) pipelines to automate orchestration of ML workflows.
	* Monitoring models, data, and infrastructure to detect issues.
	* Securing ML systems and resources through